# 02 — Shapes

Hands-on companion to [`docs/02-shapes.md`](../docs/02-shapes.md).

A shape is not just bookkeeping: it tells us which numbers are shared. We will use a small batch of examples to see how one weight vector and one bias vector can serve every row — and how their gradients gather the contributions back.

> Run this with the notebook extra installed: `pip install -e ".[notebooks]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bonsaigrad import Leaf


def show_array(ax, values, title, *, cmap="Blues"):
    """Draw a small array with its values and shape."""
    values = np.atleast_2d(values)
    ax.imshow(values, cmap=cmap, vmin=values.min() - 1, vmax=values.max() + 1)
    for (row, col), value in np.ndenumerate(values):
        ax.text(col, row, f"{value:g}", ha="center", va="center")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{title}\nshape {values.shape}", fontsize=10)


def show_vector(ax, values, title, *, color="#c1543c"):
    positions = np.arange(values.size)
    ax.bar(positions, values, color=color)
    ax.axhline(0, color="#9aa0a6", linewidth=1)
    for position, value in zip(positions, values):
        ax.text(position, value, f"{value:g}", ha="center",
                va="bottom" if value >= 0 else "top")
    ax.set_xticks(positions, [f"[{i}]" for i in positions])
    ax.set_title(f"{title}\nshape {values.shape}", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)


## One set of parameters, three examples

A layer commonly applies the same parameters to every example in a batch. Here each row of `x` is one two-feature example. `w` and `b` each have just two entries, but NumPy uses them with all three rows:

```python
scores = x * w + b
```

Before running the next cell, predict the shapes of `scores`, `w.grad`, and `b.grad`.

In [ ]:
x = Leaf([[1.0, 2.0], [-1.0, 3.0], [0.5, -2.0]])  # three examples
w = Leaf([2.0, -1.0])                              # shared weights
b = Leaf([0.5, 0.5])                               # shared bias
scores = x * w + b

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
show_array(axes[0], x.data, "examples x")
show_array(axes[1], w.data, "weights w")
show_array(axes[2], b.data, "bias b")
show_array(axes[3], scores.data, "scores = x * w + b")
plt.tight_layout()
plt.show()

print(f"x {x.data.shape} × w {w.data.shape} + b {b.data.shape} → scores {scores.data.shape}")

## Where does a shared parameter's gradient come from?

`wire()` starts with one gradient per score. Because `w` and `b` are shared across rows, each parameter receives one contribution from each example. `x` is different: each of its entries produces just one score, so its gradient stays at the batch shape.

For this expression, the contributions to `w` are the rows of `x`; the contributions to `b` are all ones; and the contribution to every row of `x` is `w`. The gradients should therefore be:

```python
w.grad = x.data.sum(axis=0)  # [0.5, 3.0]
b.grad = [3.0, 3.0]          # one contribution per row
x.grad = [[2.0, -1.0], ...]  # w, once for each example
```

In [ ]:
scores.wire()

input_contributions = np.broadcast_to(w.data, x.data.shape)
weight_contributions = x.data
bias_contributions = np.ones_like(scores.data)

fig, axes = plt.subplots(3, 3, figsize=(10, 7), width_ratios=[1.4, 1.4, 1])
show_array(axes[0, 0], scores.grad, "seeded score gradients", cmap="Reds")
show_array(axes[0, 1], input_contributions, "contributions to x.grad", cmap="Reds")
show_array(axes[0, 2], x.grad, "x.grad: one w per row", cmap="Reds")

show_array(axes[1, 0], scores.grad, "seeded score gradients", cmap="Reds")
show_array(axes[1, 1], weight_contributions, "contributions to w.grad", cmap="Reds")
show_vector(axes[1, 2], w.grad, "w.grad: sum each column")

show_array(axes[2, 0], scores.grad, "seeded score gradients", cmap="Reds")
show_array(axes[2, 1], bias_contributions, "contributions to b.grad", cmap="Reds")
show_vector(axes[2, 2], b.grad, "b.grad: count each column")

for ax in axes[1:, 1:].flat:
    ax.set_xlabel("sum over the batch axis →", color="#c1543c")
plt.tight_layout()
plt.show()

np.testing.assert_allclose(x.grad, input_contributions)
np.testing.assert_allclose(w.grad, x.data.sum(axis=0))
np.testing.assert_allclose(b.grad, [3.0, 3.0])
print("x keeps one gradient per example; shared parameters collect every row that used them.")